In [1]:
!python -V

Python 3.11.16


In [2]:
import pandas as pd
import optuna
from sklearn.model_selection import KFold, cross_val_score

In [3]:
import pickle
import pyarrow
import lightgbm as lgb

In [4]:
import seaborn as sns
import matplotlib.pyplot as plt

In [5]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge
from scipy import sparse
import numpy as np
from sklearn.metrics import root_mean_squared_error

In [6]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment('ne-experiment_oddtaxi')

<Experiment: artifact_location='/workspaces/Mlopscamp/mlruns/1', creation_time=1790098560441, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1790098560441, lifecycle_stage='active', name='ne-experiment_oddtaxi', tags={}, trace_location=None, workspace='default'>

In [7]:
def read_dataframe(filename):
    if filename.endswith('.csv'):
        df = pd.read_csv(filename)

        df['duration'] = (
            (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime'])
            .dt.total_seconds() / 60
        )
    elif filename.endswith('.parquet'):
        df = pd.read_parquet(filename)

        df['duration'] = (
            (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime'])
            .dt.total_seconds() / 60
        )
    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [8]:
df_train = read_dataframe('yellow_tripdata_2021-01.parquet')
df_val = read_dataframe('yellow_tripdata_2021-02.parquet')

In [9]:
df_train.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,duration
0,1,2021-01-01 00:30:10,2021-01-01 00:36:12,1.0,2.10,1.0,N,142,43,2,8.0,3.0,0.5,0.00,0.0,0.3,11.80,2.5,NaN,6.033333
2,1,2021-01-01 00:43:30,2021-01-01 01:11:06,1.0,14.70,1.0,N,132,165,1,42.0,0.5,0.5,8.65,0.0,0.3,51.95,0.0,NaN,27.600000
3,1,2021-01-01 00:15:48,2021-01-01 00:31:01,0.0,10.60,1.0,N,138,132,1,29.0,0.5,0.5,6.05,0.0,0.3,36.35,0.0,NaN,15.216667
4,2,2021-01-01 00:31:49,2021-01-01 00:48:21,1.0,4.94,1.0,N,68,33,1,16.5,0.5,0.5,4.06,0.0,0.3,24.36,2.5,NaN,16.533333
5,1,2021-01-01 00:16:29,2021-01-01 00:24:30,1.0,1.60,1.0,N,224,68,1,8.0,3.0,0.5,2.35,0.0,0.3,14.15,2.5,NaN,8.016667


In [10]:
len(df_train), len(df_val)

(1343254, 1340859)

In [11]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
int32_cols = df_train.select_dtypes(include=['int32']).columns
df_train[int32_cols] = df_train[int32_cols].astype('int64')
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [12]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [13]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [14]:
with mlflow.start_run():
    mlflow.set_tag('developer', "kirill")
    mlflow.log_param('train', 'yellow_tripdata_2021-01.parquet')

    lr = LinearRegression()
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)
    mlflow.log_metric('rmse',  root_mean_squared_error(y_val, y_pred))
    root_mean_squared_error(y_val, y_pred)

In [15]:
with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [16]:
import pandas as pd

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        print(name, f"{obj.memory_usage(deep=True).sum() / 1e9:.2f} GB", obj.shape)

__ 0.00 GB (5, 20)
df_train 0.25 GB (1343254, 21)
df_val 0.25 GB (1340859, 21)
_9 0.00 GB (5, 20)


In [17]:



# with mlflow.start_run():
#     mlflow.set_tag('developer', "kirill")
#     mlflow.log_param('train', 'yellow_tripdata_2021-01.parquet')
 
#     alpha = 0.0034
#     mlflow.log_param('alpha', alpha)
#     X_train.indices = X_train.indices.astype(np.int32, copy=False)
#     X_train.indptr = X_train.indptr.astype(np.int32, copy=False)
#     lr = Lasso(alpha)
#     lr.fit(X_train, y_train)

#     y_pred = lr.predict(X_val)

#     root_mean_squared_error(y_val, y_pred)
#     mlflow.log_metric('rmse',  root_mean_squared_error(y_val, y_pred))


In [18]:
# params = {
#     "num_leaves": 10000,
#     "max_depth": 100,
#     "learning_rate": 0.01,
#     "n_estimators": 10,
#     "subsample_for_bin": 100000,
# }

# # One-time data preparation, outside the run
# X_train.indices = X_train.indices.astype(np.int32, copy=False)
# X_train.indptr = X_train.indptr.astype(np.int32, copy=False)

# kf = KFold(n_splits=3, shuffle=True, random_state=42)

# with mlflow.start_run():
#     mlflow.set_tag("developer", "kirill")
#     mlflow.log_param("train", "yellow_tripdata_2021-01.parquet")
#     mlflow.log_params(params)

#     model = lgb.LGBMRegressor(**params, random_state=42, verbose=-1)
#     scores = cross_val_score(
#         model, X_train, y_train, cv=kf, scoring="neg_root_mean_squared_error"
#     )
#     rmse = -scores
#     mlflow.log_metric("rmse_mean", rmse.mean())
#     mlflow.log_metric("rmse_std", rmse.std())

In [33]:
def objective(trial):
    with mlflow.start_run():
        mlflow.set_tag('developer', "kirill")
        mlflow.log_param('train', 'yellow_tripdata_2021-01.parquet')
        num_leaves = trial.suggest_int('num_leaves', 20, 200)
        max_depth = trial.suggest_int('max_depth', -1, 100)
        learning_rate = trial.suggest_float('learning_rate', 0.05 , 0.27, log=True)
        n_estimators = trial.suggest_int('n_estimators', 100, 1000)
        subsample_for_bin = trial.suggest_int('subsample_for_bin', 100000, 300000)

        params = {
            "num_leaves": num_leaves,
            "max_depth": max_depth,
            "learning_rate": learning_rate,
            "n_estimators": n_estimators,
            "subsample_for_bin": subsample_for_bin,
        }
        mlflow.log_params(params)
        model = lgb.LGBMRegressor(**params, random_state=42, verbose=-1)
        scores = cross_val_score(
               model, X_train, y_train, cv=kf, scoring="neg_root_mean_squared_error"
               )
        rmse = -scores
        mlflow.log_metric("rmse_mean", rmse.mean())
        mlflow.log_metric("rmse_std", rmse.std())
    return rmse.mean()

In [ ]:
# def objective(trial):
#     with mlflow.start_run():
#         mlflow.set_tag('developer', "kirill")
#         mlflow.log_param('train', 'yellow_tripdata_2021-01.parquet')
#         num_leaves = trial.suggest_int('num_leaves', 20, 200)
#         max_depth = trial.suggest_int('max_depth', -1, 100)
#         learning_rate = trial.suggest_float('learning_rate', 0.05 , 0.27, log=True)
#         n_estimators = trial.suggest_int('n_estimators', 100, 1000)
#         subsample_for_bin = trial.suggest_int('subsample_for_bin', 100000, 300000)
#         train_set = lgb.Dataset(X_train, label=y_train)
#         params = {
#             "num_leaves": num_leaves,
#             "max_depth": max_depth,
#             "metric": "rmse",
#             "learning_rate": learning_rate,
#             "verbose": -1
#         }
#         mlflow.log_params(params)
#         cv_results = lgb.cv(
#                 params,
#                 train_set,
#                 num_boost_round=5000,
#                 folds=kf,
#                 stratified=False,
#                 callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
#             )
#         rmse_mean = cv_results["valid rmse-mean"][-1]
#         mlflow.log_metric("rmse_mean",  cv_results["valid rmse-mean"][-1])
#         mlflow.log_metric("rmse_std",  cv_results["valid rmse-stdv"][-1])
#     return  rmse_mean

In [34]:
X_train.indices = X_train.indices.astype(np.int32, copy=False)
X_train.indptr = X_train.indptr.astype(np.int32, copy=False)

kf = KFold(n_splits=3, shuffle=True, random_state=42)
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.RandomSampler(seed=42))
study.optimize(objective, n_trials=60)

[I 2026-09-27 17:07:02,805] A new study created in memory with name: no-name-120b0e87-ee20-4785-85f5-a80f54ddea5b
[I 2026-09-27 17:11:04,894] Trial 0 finished with value: 3.996665240832742 and parameters: {'num_leaves': 87, 'max_depth': 95, 'learning_rate': 0.17182161609647528, 'n_estimators': 639, 'subsample_for_bin': 131203}. Best is trial 0 with value: 3.996665240832742.
[W 2026-09-27 17:11:17,474] Trial 1 failed with parameters: {'num_leaves': 48, 'max_depth': 4, 'learning_rate': 0.2154527334764423, 'n_estimators': 641, 'subsample_for_bin': 241615} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/opt/conda/envs/mlflow/lib/python3.11/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_32663/1493598629.py", line 20, in objective
    scores = cross_val_score(
             ^^^^^^^^^^^^^^^^
  File "/opt/conda/envs/mlflow/lib/python3.11/

KeyboardInterrupt: 